In [34]:
from typing import TypedDict
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langchain.messages import SystemMessage, HumanMessage
from dataclasses import dataclass, field
from pathlib import Path

# Config

# Init setup

In [ ]:
llm = init_chat_model()#TODO make work

class Verification(BaseModel):
    verified: list[str] # claims that are verified to hold in the ontology
    violations: list[str] # claims where a contradiction or counter example exists
    not_verifiable: list[str] # claims that hold only due to the open world principle

@dataclass
class State:
    user_prompt: str
    tweets: Path
    summary_count: int = 0
    summary: str = ""
    verified: list[str] = field(default_factory=list)
    violations: list[str] = field(default_factory=list)
    not_verifiable: list[str] = field(default_factory=list)
    history: State = None
    # ^turns state into a linked list, bad for performance but will be fine for small sizes like here


# Nodes (Agentics and deterministic)

## Sumarization Agent

In [36]:
summarization_system_prompt = """TODO""" #TODO

In [37]:
def summarize(state: State):
    
    # send prompt to LLM (no agent needed, this is a single linear call)
    response = llm.invoke([
        SystemMessage(summarization_system_prompt),
        HumanMessage(build_prompt(state.user_prompt, state.tweets))
    ])

    # make new state, storing historical state, incrementing the sumarycount and adding the LLM response
    return State(state.user_prompt, state.tweets, state.summary_count+1, response.content, history = state)

In [38]:
def build_prompt(user_prompt: str, tweets: Path):
    """"Assumes tweets is a complete filepath"""
    with open(tweets, 'r') as f: #TODO wierd issue with file format (utf8 doesn't work)
        return f"{user_prompt}\n{f.readlines()}"

## Verification Agent

In [39]:
verification_system_prompt = """TODO""" #TODO

In [40]:
@tool
def check_ontology(query: str):
    "TODO"
    pass #TODO
    # need to catch actual error and return them as text for the agent to retry failed queries
    

In [41]:
checker_agent = create_agent(
    llm,
    tools=[check_ontology],
    system_prompt=verification_system_prompt,
    response_format=Verification
)

In [42]:
def verify(state: State):
    result = checker_agent.invoke({'messages': [
        SystemMessage(verification_system_prompt),
        HumanMessage(state.summary)
    ]})
    v: Verification = result['structured_response']
    return v # may need explicit cast to state

## Best summary picker

In [43]:
def pick_best_summary(state: State):
    def helper(current_state: State, best_state: State) -> State:
        if current_state.history is None:
            return best_state
        if len(current_state.violations) <= len(best_state.violations):
            return helper(current_state.history, current_state)
        else:
            return helper(current_state.history, best_state)
    return helper(state, state)

# recursive due to linked list, bit akward but it should work

# Routing Functions

In [44]:
def route(state: State, max_tries: int = 3): #TODO rename
    "both routing diamonds in fig 1"
    if len(state.violations) == 0:
        return END
    elif state.summary_count < max_tries:
        return 'summarize'
    else:
        return 'pick_best_summary'

# Orchestration

In [45]:
graph = StateGraph(State)
graph.add_node('summarize', summarize)
graph.add_node('verify_summary', verify)
graph.add_node('pick_best_summary', pick_best_summary)
graph.add_edge(START, 'summarize')
graph.add_edge('summarize', 'verify_summary')
graph.add_conditional_edges('verify_summary', route)
graph.add_edge('pick_best_summary', END)

app = graph.compile()


# Final call to run

In [46]:
init_state = State(
    user_prompt='todo',
    tweets=Path('inputs/Netflix_clean.txt'))
out = app.invoke(init_state)

TypeError: _init_chat_model_helper() missing 1 required positional argument: 'model'

In [ ]:
out

{'user_prompt': 'todo',
 'tweets': 'todo',
 'i': 0,
 'summary': '',
 'verified': [],
 'violations': [],
 'not_verifiable': [],
 'history': []}